# 과제 2 · 시각화 실습 — 그림으로 보고, 검정으로 확인한다 (6주차 · 2026. 10. 19.)

기초지질데이터과학 · **제출: 오늘 23:59, LMS 과제 2 — 이 노트북(.ipynb) 파일 하나** · 배점 5 (코드 작동 2 · 해석 2 · 기록지 1)

**이름:**　　　　　　**학번:**

- 쓰는 파일: `rayleigh_nitrogen.csv`, `sandstone_two_units.csv` — 왼쪽 📁 파일 창에 끌어다 놓으세요.
- `rayleigh_nitrogen.csv`는 **실제 측정 자료**입니다 — 레일리 경(Lord Rayleigh)이 1893–94년에 잰 질소 질량 15회, 강의 27쪽의 표. `sandstone_two_units.csv`는 **교육용 합성 자료**입니다.
- AI는 자유롭게 씁니다. 쓴 것은 맨 아래 **기록지**에 적습니다. 검증 없는 제출은 해석 0점입니다.
- 오늘의 순서: **그림으로 보고 → 검정으로 확인하고 → 다시 그림을 본다.** 그림의 인상과 검정 결과가 다르면, 어느 쪽이 무엇 때문에 틀렸는지 찾습니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
try:                                   # 그림의 한글 글꼴 (Colab에서 처음 한 번 설치, 5초쯤)
    import koreanize_matplotlib
except ImportError:
    !pip -q install koreanize-matplotlib
    import koreanize_matplotlib

def summary(x, name):
    # 한 집단 요약: 개수 · 평균 · 표준편차(SD) · 표준오차(SE = SD/√n) · 평균의 95% 신뢰구간(t 분포, 강의 156쪽)
    x = pd.Series(x, dtype=float).dropna().to_numpy()
    n, m, s = len(x), x.mean(), x.std(ddof=1)
    se = s / np.sqrt(n)
    h = stats.t.ppf(0.975, n - 1) * se
    print(f"{name:<14} n={n:2d} | 평균 {m:.5f} | SD {s:.5f} | SE {se:.5f} | 95% CI [{m-h:.5f}, {m+h:.5f}]")

def welch(x, y, name):
    # 두 표본 Welch t-검정 — 두 집단의 분산이 같다고 가정하지 않는다(강의 179쪽). 차이 = x 평균 − y 평균
    x = pd.Series(x, dtype=float).dropna().to_numpy()
    y = pd.Series(y, dtype=float).dropna().to_numpy()
    vx, vy = x.var(ddof=1) / len(x), y.var(ddof=1) / len(y)
    t = (x.mean() - y.mean()) / np.sqrt(vx + vy)
    df = (vx + vy) ** 2 / (vx ** 2 / (len(x) - 1) + vy ** 2 / (len(y) - 1))
    p = 2 * stats.t.sf(abs(t), df)
    tc = stats.t.ppf(0.975, df)
    ptxt = f"= {p:.4f}" if p >= 1e-4 else "< 0.0001"
    print(f"{name:<14} 차이 {x.mean()-y.mean():+.5f} | t = {t:6.2f} | df = {df:4.1f} | 임계 t(α=0.05) = {tc:.2f} | p {ptxt}")

## Part A · 레일리의 질소 — 막대가 가린 차이 (약 20분)

1893–94년 겨울, 레일리 경은 질소를 표준 유리구에 채워 질량을 쟀습니다(강의 26–32쪽). **공기**에서 산소를 걷어 내 얻은 질소와 **화학 반응**(NO · N₂O · NH₄NO₂의 분해)으로 얻은 질소를 비교했고, 이 차이가 이듬해 **아르곤 발견**으로 이어졌습니다. 26쪽의 질문 — *"그는 어떻게 동료들을 설득했을까?"*

열: `측정일` · `원료` · `정제제`(산소를 걷어 낸 방법) · `질량_g`

In [ ]:
ra = pd.read_csv("rayleigh_nitrogen.csv")
ra["출처"] = np.where(ra["원료"] == "공기", "공기", "화학 반응")
ra

**A-1. AI 그림.** AI에게 아래처럼 요청하고, 받은 코드를 다음 칸에 붙여 넣어 **고치지 말고** 실행하세요.

> pandas 데이터프레임 `ra`에서 `출처`(공기 · 화학 반응)별 `질량_g`를 비교하는 그림을 matplotlib으로 그려 줘.

**그림만 보고 내린 판단 (한 줄):**

In [ ]:
# A-1: AI가 준 코드

**A-2. 내 그림.** 원자료 점과 상자그림을 함께 그리세요. 세로축 범위는 자료가 있는 곳으로(31쪽), 축 이름에는 단위를 적습니다. AI에게 시켜도 되지만, 무엇을 바꿨는지는 기록지에 남기세요.

In [ ]:
# A-2: 내 그림 (원자료 점 + 상자그림)

**A-3. 검정.** 아래 칸을 실행하세요. 두 출처의 차이는 질량의 몇 %인가요? 계산한 t는 임계 t의 몇 배인가요?

In [ ]:
air = ra.loc[ra["출처"] == "공기", "질량_g"]
chem = ra.loc[ra["출처"] == "화학 반응", "질량_g"]
summary(air, "공기")
summary(chem, "화학 반응")
welch(air, chem, "공기 − 화학")

**A-4. 해석 (각각 한두 문장).**

1. A-1의 그림(또는 30쪽의 막대그래프)은 왜 이 차이를 보여 주지 못했나요? A-2의 그림은 무엇을 바꿔서 보여 주나요? (A-1 그림이 이미 차이를 잘 보여 줬다면, 그 그림이 막대그래프와 무엇이 다른지 쓰세요.)
2. 차이가 **원료**가 아니라 **정제제** 때문일 수는 없나요? 정제제가 같은 시료끼리만 비교해 보고 답하세요. (화학 반응 질소는 모두 '뜨거운 철'로 정제했습니다.)

**1:**

**2:**

In [ ]:
# A-4 ②: 정제제가 같은 시료끼리 비교

## Part B · 두 사암층의 밀도 — 그림의 인상과 검정 (약 35분)

**질문:** 한 지역의 하부 **A층**과 상부 **B층** 사암에서 시료를 채취해 겉보기 밀도(g/cm³)를 쟀습니다. **두 층의 밀도는 다른가?** 이 값은 층별 밀도로 이 지역 중력 이상 해석(지하 구조 모델)에 그대로 들어갑니다.

열: `시료번호` · `층` · `채취일` · `밀도`(g/cm³) · `비고`(현장 관찰). 사암 겉보기 밀도는 대략 2.0–2.6 g/cm³이고, 석영 2.65 · 방해석 2.71보다 클 수 없습니다.

In [ ]:
ss = pd.read_csv("sandstone_two_units.csv")
print(ss.groupby("층")["밀도"].describe().round(3))
ss

**B-1. AI가 흔히 그려 주는 그림.** 아래 코드는 AI에게 "두 층의 밀도를 비교하는 그림을 그려 줘"라고 했을 때 흔히 받는 형태를 그대로 옮긴 것입니다. **고치지 말고** 실행하세요. 그림의 인상을 한 줄로 적고, 1교시 **점검표 다섯 항목**으로 문제를 찾아 적으세요.

**그림의 인상 (한 줄):**

**점검표로 찾은 문제:**

In [ ]:
# B-1: AI가 흔히 내놓는 그림 — 고치지 말고 실행
g = ss.groupby("층")["밀도"]
means, sems = g.mean(), g.sem()
plt.figure(figsize=(4.5, 4))
plt.bar(means.index, means.values, yerr=sems.values, capsize=8, color=["skyblue", "salmon"])
plt.ylim(2.25, 2.37)
plt.ylabel("밀도")
plt.title("B층이 A층보다 뚜렷하게 무겁다")
plt.show()

**B-2. 내 그림.** 층마다 **원자료 점 + 상자그림 + 평균의 95% 신뢰구간**을 함께 그리세요(1교시의 '세 겹'). 축 이름과 단위, 오차막대가 무엇인지를 그림 안(범례·제목)에 적습니다. **비고가 있는 시료**가 그림에서 어디에 있는지 확인하세요.

In [ ]:
# B-2: 내 그림 (원자료 점 + 상자그림 + 평균의 95% 신뢰구간)

**B-3. 검정과 결정.** 먼저 자료 전체로 검정합니다. 그다음 B-2 그림과 비고를 보고, **질문에 비추어** 다르게 다룰 시료가 있는지 정하세요(뺌 · 남김 · 다른 방법). 그 결정대로 다시 검정합니다. 근거는 자료 **밖**에서 옵니다 — 비고, 암석의 물리, 그리고 질문(층의 밀도를 무엇에 쓰나).

| 시료번호 | 값 | 결정 | 근거 (AI 밖의 기준) |
|---|---|---|---|
| | | | |

In [ ]:
A = ss.loc[ss["층"] == "A층(하부)", "밀도"]
B = ss.loc[ss["층"] == "B층(상부)", "밀도"]
summary(A, "A층")
summary(B, "B층 전체")
welch(B, A, "B − A 전체")

# 결정대로 다시 — 아래 B_my 한 줄을 결정에 맞게 고치세요
B_my = B.copy()
summary(B_my, "B층 결정 후")
welch(B_my, A, "B − A 결정 후")

**B-4. 결론.** 보조 ②의 틀로 쓰세요.

> 〔어느 자료〕에서 〔무엇을〕 보니 〔수치와 방향 — 차이, p 또는 신뢰구간〕이었다. 이는 〔지질학적 해석〕으로 설명할 수 있다. 다만 〔한계〕 때문에 〔여기까지〕만 말할 수 있다.

**결론:**

**B-1 그림의 인상은 검정 결과와 맞았나요? 맞지 않았다면 그림의 어떤 선택 때문인가요? (한두 문장):**

**B-5. 질문이 바뀌면? (두세 줄)** 질문을 "**풍화가 이 사암의 밀도를 얼마나 낮추나?**"로 바꾸면, 오늘 내린 결정과 그림·검정에서 무엇이 달라지나요? 한 가지 이상.

**답:**

## 짝 상호평가 (약 10분) — 참여 ②

짝과 화면을 바꿔 **짝의 B-2 그림**을 아래 점검표로 평가하고, 평가는 **내 노트북**의 이 표에 적습니다. 그다음 짝에게 받은 지적으로 **내 그림을 고쳐** B-2 아래에 새 칸으로 붙입니다(수정본).

**평가한 짝 (이름):**

| # | 점검 항목 | ○ / △ / ✕ | 한 줄 이유 |
|---|---|---|---|
| ① | 원자료(점)가 보이는가 | | |
| ② | 축 범위가 차이를 숨기거나 키우지 않는가 | | |
| ③ | 오차막대가 무엇인지(SD · SE · 95% CI) 적혀 있는가 | | |
| ④ | 축 이름 · 단위 · 범례가 있는가 | | |
| ⑤ | 그림의 인상이 검정 결과와 맞는가 | | |

**짝에게 받은 지적과 고친 점 (한두 줄):**

## AI 활용 기록지 (5칸) — 최소 2건 (A-1 포함)

| # | ① 도구·모델 | ② 프롬프트 원문 | ③ AI 산출물 (셀 번호·요약) | ④ 검증 — 무엇을 보고 확인했나 | ⑤ 수정 — 무엇을 왜 바꿨나 |
|---|---|---|---|---|---|
| 1 | | | | | |
| 2 | | | | | |

**④칸 예시.** "막대그래프의 세로축이 0에서 시작하는지 확인했다" · "오차막대가 무엇인지 코드에서 확인했다 — `sem()`은 표준오차" · "summary()의 평균을 값 몇 개로 손계산해 대조했다" · "그림에서 튀는 점이 표의 어느 시료인지 확인했다". **"AI에게 다시 물어봤다"는 검증이 아닙니다.**